In [5]:
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
# Training data
X_train = joblib.load("../data/X_train_zero_day.pkl")
y_train = joblib.load("../data/y_train_zero_day.pkl")

# Seen attacks
X_test_seen = joblib.load("../data/X_test_seen.pkl")
y_test_seen = joblib.load("../data/y_test_seen.pkl")

# Zero-Day attacks
X_test_zero_day = joblib.load("../data/X_test_zero_day.pkl")
y_test_zero_day = joblib.load("../data/y_test_zero_day.pkl")

# Scaled data
X_train_scaled = joblib.load("../data/X_train_zero_day_scaled.pkl")
X_test_seen_scaled = joblib.load("../data/X_test_seen_scaled.pkl")
X_test_zero_day_scaled = joblib.load("../data/X_test_zero_day_scaled.pkl")

print("Data loaded successfully.")

print("X_train:", X_train.shape)
print("X_test_seen:", X_test_seen.shape)
print("X_test_zero_day:", X_test_zero_day.shape)

Data loaded successfully.
X_train: (125973, 122)
X_test_seen: (18794, 122)
X_test_zero_day: (3750, 122)


In [7]:
columns = [
    'duration',
    'protocol_type',
    'service',
    'flag',
    'src_bytes',
    'dst_bytes',
    'land',
    'wrong_fragment',
    'urgent',
    'hot',
    'num_failed_logins',
    'logged_in',
    'num_compromised',
    'root_shell',
    'su_attempted',
    'num_root',
    'num_file_creations',
    'num_shells',
    'num_access_files',
    'num_outbound_cmds',
    'is_host_login',
    'is_guest_login',
    'count',
    'srv_count',
    'serror_rate',
    'srv_serror_rate',
    'rerror_rate',
    'srv_rerror_rate',
    'same_srv_rate',
    'diff_srv_rate',
    'srv_diff_host_rate',
    'dst_host_count',
    'dst_host_srv_count',
    'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate',
    'dst_host_srv_serror_rate',
    'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate',
    'label',
    'difficulty'
]

test_raw = pd.read_csv(
    "../data/KDDTest+.txt",
    names=columns
)

print("Original test dataset:", test_raw.shape)

Original test dataset: (22544, 43)


In [8]:
# Load training labels in their original string form
train_raw = pd.read_csv(
    "../data/KDDTrain+.txt",
    names=columns
)

# Attack types present in training dataset
train_attack_labels = set(
    train_raw.loc[
        train_raw["label"] != "normal",
        "label"
    ].unique()
)

# Attack types present in test dataset
test_attack_labels = set(
    test_raw.loc[
        test_raw["label"] != "normal",
        "label"
    ].unique()
)

# Attack types that never appeared in training
zero_day_labels = sorted(
    test_attack_labels - train_attack_labels
)

print("Zero-Day attack types:")
print(zero_day_labels)

print("\nNumber of Zero-Day attack types:", len(zero_day_labels))

Zero-Day attack types:
['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']

Number of Zero-Day attack types: 17


In [9]:
test_zero_day_df = test_raw[
    test_raw["label"].isin(zero_day_labels)
].copy()

test_zero_day_labels = test_zero_day_df["label"].values

print("Number of Zero-Day samples:", len(test_zero_day_labels))

print("\nZero-Day label distribution:")
print(
    pd.Series(test_zero_day_labels)
    .value_counts()
    .sort_index()
)

Number of Zero-Day samples: 3750

Zero-Day label distribution:
apache2          737
httptunnel       133
mailbomb         293
mscan            996
named             17
processtable     685
ps                15
saint            319
sendmail          14
snmpgetattack    178
snmpguess        331
sqlattack          2
udpstorm           2
worm               2
xlock              9
xsnoop             4
xterm             13
Name: count, dtype: int64


In [10]:
assert len(test_zero_day_labels) == len(X_test_zero_day), (
    f"Mismatch! Labels: {len(test_zero_day_labels)}, "
    f"Samples: {len(X_test_zero_day)}"
)

assert len(y_test_zero_day) == len(X_test_zero_day)

print("Validation passed.")
print("Zero-Day samples:", len(X_test_zero_day))
print("Zero-Day labels:", len(test_zero_day_labels))

Validation passed.
Zero-Day samples: 3750
Zero-Day labels: 3750


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

In [13]:
best_dt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=40,
    min_samples_split=20,
    min_samples_leaf=1,
    max_features=None,
    random_state=42
)

best_dt.fit(
    X_train,
    y_train
)

print("Tuned Decision Tree trained.")

Tuned Decision Tree trained.


In [14]:
best_rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=40,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(
    X_train,
    y_train
)

print("Tuned Random Forest trained.")

Tuned Random Forest trained.


In [ ]:
best_knn = KNeighborsClassifier(
    n_neighbors=3,
    weights="distance",
    metric="manhattan",
    n_jobs=-1
)

best_knn.fit(
    X_train_scaled,
    y_train
)

print("Tuned KNN trained.")


Tuned KNN trained.
KNeighborsClassifier(metric='manhattan', n_jobs=-1, n_neighbors=3,
                     weights='distance')


In [28]:
from sklearn.svm import LinearSVC

best_svm = LinearSVC(
    C=0.5,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

best_svm.fit(
    X_train_scaled,
    y_train
)

print("Tuned Linear SVM trained successfully.")

Tuned Linear SVM trained successfully.


In [16]:
best_logreg = LogisticRegression(
    C=10,
    class_weight=None,
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

best_logreg.fit(
    X_train_scaled,
    y_train
)

print("Tuned Logistic Regression trained.")

Tuned Logistic Regression trained.


In [29]:
# Decision Tree
y_pred_zero_day_dt_tuned = best_dt.predict(
    X_test_zero_day
)

# Random Forest
y_pred_zero_day_rf_tuned = best_rf.predict(
    X_test_zero_day
)

# KNN
y_pred_zero_day_knn_tuned = best_knn.predict(
    X_test_zero_day_scaled
)

# SVM
y_pred_zero_day_svm_tuned = best_svm.predict(
    X_test_zero_day_scaled
)

# Logistic Regression
y_pred_zero_day_logreg_tuned = best_logreg.predict(
    X_test_zero_day_scaled
)

print("Predictions generated successfully.")

Predictions generated successfully.


In [30]:
predictions = {
    "Decision Tree": y_pred_zero_day_dt_tuned,
    "Random Forest": y_pred_zero_day_rf_tuned,
    "KNN": y_pred_zero_day_knn_tuned,
    "Linear SVM": y_pred_zero_day_svm_tuned,
    "Logistic Regression": y_pred_zero_day_logreg_tuned
}

for model_name, predictions_array in predictions.items():

    assert len(predictions_array) == len(test_zero_day_labels), (
        f"{model_name}: prediction length mismatch"
    )

    print(
        f"{model_name}: "
        f"{len(predictions_array)} predictions ✓"
    )

Decision Tree: 3750 predictions ✓
Random Forest: 3750 predictions ✓
KNN: 3750 predictions ✓
Linear SVM: 3750 predictions ✓
Logistic Regression: 3750 predictions ✓


In [31]:
def calculate_per_attack_detection_rate(
    true_labels,
    predictions
):
    
    results = []

    unique_attacks = sorted(
        pd.Series(true_labels).unique()
    )

    for attack in unique_attacks:

        mask = np.array(true_labels) == attack

        attack_predictions = np.array(predictions)[mask]

        total_samples = len(attack_predictions)

        detected_samples = np.sum(
            attack_predictions == 1
        )

        detection_rate = (
            detected_samples / total_samples * 100
        )

        results.append({
            "Attack Type": attack,
            "Total Samples": total_samples,
            "Detected": detected_samples,
            "Missed": total_samples - detected_samples,
            "Detection Rate (%)": detection_rate
        })

    return pd.DataFrame(results)

In [32]:
def calculate_per_attack_detection_rate(
    true_labels,
    predictions
):
    
    results = []

    unique_attacks = sorted(
        pd.Series(true_labels).unique()
    )

    for attack in unique_attacks:

        mask = np.array(true_labels) == attack

        attack_predictions = np.array(predictions)[mask]

        total_samples = len(attack_predictions)

        detected_samples = np.sum(
            attack_predictions == 1
        )

        detection_rate = (
            detected_samples / total_samples * 100
        )

        results.append({
            "Attack Type": attack,
            "Total Samples": total_samples,
            "Detected": detected_samples,
            "Missed": total_samples - detected_samples,
            "Detection Rate (%)": detection_rate
        })

    return pd.DataFrame(results)

In [33]:
dt_attack_results = calculate_per_attack_detection_rate(
    test_zero_day_labels,
    y_pred_zero_day_dt_tuned
)

rf_attack_results = calculate_per_attack_detection_rate(
    test_zero_day_labels,
    y_pred_zero_day_rf_tuned
)

knn_attack_results = calculate_per_attack_detection_rate(
    test_zero_day_labels,
    y_pred_zero_day_knn_tuned
)

svm_attack_results = calculate_per_attack_detection_rate(
    test_zero_day_labels,
    y_pred_zero_day_svm_tuned
)

logreg_attack_results = calculate_per_attack_detection_rate(
    test_zero_day_labels,
    y_pred_zero_day_logreg_tuned
)

In [34]:
per_attack_comparison = pd.DataFrame({
    "Attack Type": dt_attack_results["Attack Type"],
    "Samples": dt_attack_results["Total Samples"],
    
    "Decision Tree (%)":
        dt_attack_results["Detection Rate (%)"],
    
    "Random Forest (%)":
        rf_attack_results["Detection Rate (%)"],
    
    "KNN (%)":
        knn_attack_results["Detection Rate (%)"],
    
    "Linear SVM (%)":
        svm_attack_results["Detection Rate (%)"],
    
    "Logistic Regression (%)":
        logreg_attack_results["Detection Rate (%)"]
})

per_attack_comparison

,Attack Type,Samples,Decision Tree (%),Random Forest (%),KNN (%),Linear SVM (%),Logistic Regression (%)
0,apache2,737,66.621438,61.465400,54.816825,99.185889,99.050204
1,httptunnel,133,12.781955,60.902256,14.285714,1.503759,1.503759
2,mailbomb,293,0.682594,0.000000,0.000000,0.000000,0.000000
3,mscan,996,52.510040,57.831325,77.208835,34.236948,49.196787
4,named,17,17.647059,29.411765,35.294118,5.882353,5.882353
5,processtable,685,2.627737,30.218978,1.313869,0.437956,0.291971
6,ps,15,13.333333,53.333333,20.000000,0.000000,0.000000
7,saint,319,97.178683,99.373041,97.492163,95.297806,94.984326
8,sendmail,14,0.000000,0.000000,7.142857,7.142857,7.142857
9,snmpgetattack,178,0.000000,0.000000,0.561798,0.561798,0.561798


In [35]:
percentage_columns = [
    "Decision Tree (%)",
    "Random Forest (%)",
    "KNN (%)",
    "Linear SVM (%)",
    "Logistic Regression (%)"
]

per_attack_comparison[
    percentage_columns
] = per_attack_comparison[
    percentage_columns
].round(2)

per_attack_comparison

,Attack Type,Samples,Decision Tree (%),Random Forest (%),KNN (%),Linear SVM (%),Logistic Regression (%)
0,apache2,737,66.62,61.47,54.82,99.19,99.05
1,httptunnel,133,12.78,60.90,14.29,1.50,1.50
2,mailbomb,293,0.68,0.00,0.00,0.00,0.00
3,mscan,996,52.51,57.83,77.21,34.24,49.20
4,named,17,17.65,29.41,35.29,5.88,5.88
5,processtable,685,2.63,30.22,1.31,0.44,0.29
6,ps,15,13.33,53.33,20.00,0.00,0.00
7,saint,319,97.18,99.37,97.49,95.30,94.98
8,sendmail,14,0.00,0.00,7.14,7.14,7.14
9,snmpgetattack,178,0.00,0.00,0.56,0.56,0.56


In [36]:
per_attack_comparison = (
    per_attack_comparison
    .sort_values("Attack Type")
    .reset_index(drop=True)
)

per_attack_comparison

,Attack Type,Samples,Decision Tree (%),Random Forest (%),KNN (%),Linear SVM (%),Logistic Regression (%)
0,apache2,737,66.62,61.47,54.82,99.19,99.05
1,httptunnel,133,12.78,60.90,14.29,1.50,1.50
2,mailbomb,293,0.68,0.00,0.00,0.00,0.00
3,mscan,996,52.51,57.83,77.21,34.24,49.20
4,named,17,17.65,29.41,35.29,5.88,5.88
5,processtable,685,2.63,30.22,1.31,0.44,0.29
6,ps,15,13.33,53.33,20.00,0.00,0.00
7,saint,319,97.18,99.37,97.49,95.30,94.98
8,sendmail,14,0.00,0.00,7.14,7.14,7.14
9,snmpgetattack,178,0.00,0.00,0.56,0.56,0.56


In [37]:
mean_detection_rate = (
    per_attack_comparison[
        percentage_columns
    ]
    .mean()
    .sort_values(ascending=False)
)

mean_detection_rate

Random Forest (%)          37.230000
KNN (%)                    27.870000
Decision Tree (%)          25.309412
Logistic Regression (%)    16.151765
Linear SVM (%)             15.307647
dtype: float64

In [38]:
overall_detection_rate = {}

for model_name, predictions_array in predictions.items():

    detected = np.sum(
        predictions_array == 1
    )

    total = len(predictions_array)

    overall_detection_rate[model_name] = (
        detected / total * 100
    )

overall_detection_df = (
    pd.DataFrame.from_dict(
        overall_detection_rate,
        orient="index",
        columns=["Overall Zero-Day Detection Rate (%)"]
    )
    .sort_values(
        "Overall Zero-Day Detection Rate (%)",
        ascending=False
    )
)

overall_detection_df

,Overall Zero-Day Detection Rate (%)
Random Forest,44.106667
Logistic Regression,40.906667
KNN,40.773333
Linear SVM,37.013333
Decision Tree,36.640000


In [39]:
detailed_results = pd.DataFrame({
    "Attack Type": test_zero_day_labels
})

detailed_results["DT"] = y_pred_zero_day_dt_tuned
detailed_results["RF"] = y_pred_zero_day_rf_tuned
detailed_results["KNN"] = y_pred_zero_day_knn_tuned
detailed_results["SVM"] = y_pred_zero_day_svm_tuned
detailed_results["LR"] = y_pred_zero_day_logreg_tuned

detailed_results.head()

,Attack Type,DT,RF,KNN,SVM,LR
0,saint,1,1,1,1,1
1,mscan,0,0,0,0,0
2,mscan,0,1,0,0,0
3,mscan,1,1,1,1,1
4,apache2,1,1,1,1,1


In [41]:
per_attack_comparison.to_csv(
    "../results/per_attack_tuned_models.csv",
    index=False
)

overall_detection_df.to_csv(
    "../results/overall_zero_day_detection_tuned.csv"
)

print("Results saved successfully.")

Results saved successfully.


In [42]:
model_columns = [
    "Decision Tree (%)",
    "Random Forest (%)",
    "KNN (%)",
    "Linear SVM (%)",
    "Logistic Regression (%)"
]

per_attack_comparison["Best Model"] = (
    per_attack_comparison[model_columns]
    .idxmax(axis=1)
)

per_attack_comparison[
    [
        "Attack Type",
        "Samples",
        *model_columns,
        "Best Model"
    ]
]

,Attack Type,Samples,Decision Tree (%),Random Forest (%),KNN (%),Linear SVM (%),Logistic Regression (%),Best Model
0,apache2,737,66.62,61.47,54.82,99.19,99.05,Linear SVM (%)
1,httptunnel,133,12.78,60.90,14.29,1.50,1.50,Random Forest (%)
2,mailbomb,293,0.68,0.00,0.00,0.00,0.00,Decision Tree (%)
3,mscan,996,52.51,57.83,77.21,34.24,49.20,KNN (%)
4,named,17,17.65,29.41,35.29,5.88,5.88,KNN (%)
5,processtable,685,2.63,30.22,1.31,0.44,0.29,Random Forest (%)
6,ps,15,13.33,53.33,20.00,0.00,0.00,Random Forest (%)
7,saint,319,97.18,99.37,97.49,95.30,94.98,Random Forest (%)
8,sendmail,14,0.00,0.00,7.14,7.14,7.14,KNN (%)
9,snmpgetattack,178,0.00,0.00,0.56,0.56,0.56,KNN (%)


In [43]:
best_model_counts = (
    per_attack_comparison["Best Model"]
    .value_counts()
)

best_model_counts

Best Model
Decision Tree (%)    6
Random Forest (%)    5
KNN (%)              4
Linear SVM (%)       2
Name: count, dtype: int64

In [44]:
zero_detection = {}

for model_name, predictions_array in predictions.items():

    attack_results = calculate_per_attack_detection_rate(
        test_zero_day_labels,
        predictions_array
    )

    zero_attacks = attack_results[
        attack_results["Detection Rate (%)"] == 0
    ]["Attack Type"].tolist()

    zero_detection[model_name] = zero_attacks


for model_name, attacks in zero_detection.items():

    print(f"\n{model_name}")
    print("-" * 40)

    if len(attacks) == 0:
        print("No attack type had 0% detection.")
    else:
        print(attacks)


Decision Tree
----------------------------------------
['sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'worm']

Random Forest
----------------------------------------
['mailbomb', 'sendmail', 'snmpgetattack', 'snmpguess', 'worm', 'xlock']

KNN
----------------------------------------
['mailbomb', 'worm', 'xlock', 'xsnoop']

Linear SVM
----------------------------------------
['mailbomb', 'ps', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop']

Logistic Regression
----------------------------------------
['mailbomb', 'ps', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop']


In [45]:
per_attack_comparison["Mean Detection (%)"] = (
    per_attack_comparison[model_columns]
    .mean(axis=1)
)

hardest_attacks = (
    per_attack_comparison[
        [
            "Attack Type",
            "Samples",
            "Mean Detection (%)"
        ]
    ]
    .sort_values(
        "Mean Detection (%)"
    )
)

hardest_attacks

,Attack Type,Samples,Mean Detection (%)
13,worm,2,0.000
2,mailbomb,293,0.136
10,snmpguess,331,0.300
9,snmpgetattack,178,0.336
14,xlock,9,2.222
8,sendmail,14,4.284
5,processtable,685,6.978
15,xsnoop,4,10.000
6,ps,15,17.332
1,httptunnel,133,18.194


In [46]:
easiest_attacks = (
    per_attack_comparison[
        [
            "Attack Type",
            "Samples",
            "Mean Detection (%)"
        ]
    ]
    .sort_values(
        "Mean Detection (%)",
        ascending=False
    )
)

easiest_attacks

,Attack Type,Samples,Mean Detection (%)
7,saint,319,96.864
0,apache2,737,76.230
3,mscan,996,54.198
12,udpstorm,2,50.000
11,sqlattack,2,40.000
4,named,17,18.822
16,xterm,13,18.458
1,httptunnel,133,18.194
6,ps,15,17.332
15,xsnoop,4,10.000


In [47]:
final_summary = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "KNN",
        "Linear SVM",
        "Logistic Regression"
    ],
    
    "Overall Zero-Day Detection (%)": [
        overall_detection_rate["Decision Tree"],
        overall_detection_rate["Random Forest"],
        overall_detection_rate["KNN"],
        overall_detection_rate["Linear SVM"],
        overall_detection_rate["Logistic Regression"]
    ],
    
    "Mean Per-Attack Detection (%)": [
        mean_detection_rate["Decision Tree (%)"],
        mean_detection_rate["Random Forest (%)"],
        mean_detection_rate["KNN (%)"],
        mean_detection_rate["Linear SVM (%)"],
        mean_detection_rate["Logistic Regression (%)"]
    ]
})

final_summary = final_summary.round(2)

final_summary

,Model,Overall Zero-Day Detection (%),Mean Per-Attack Detection (%)
0,Decision Tree,36.64,25.31
1,Random Forest,44.11,37.23
2,KNN,40.77,27.87
3,Linear SVM,37.01,15.31
4,Logistic Regression,40.91,16.15


In [48]:
final_summary.to_csv(
    "../results/final_zero_day_tuned_summary.csv",
    index=False
)

print("Final summary saved.")

Final summary saved.
